# 09_00 EES Uncertainty and Rank Stability — COMPAS

## Reviewer-facing purpose
This notebook addresses the remaining uncertainty gap identified by the reviewers: **are the EES component scores, composite EES values, and resulting model rankings sensitive to the particular 80/20 train-test partition used in the primary analysis?**

It performs **20 repeated stratified 80/20 holdout evaluations**. Only the train-test partition changes. Model families, fixed hyperparameters, EES definitions, corrected Regulatory Alignment, and primary equal EES weights remain unchanged.

This first version is configured for **German Credit**. It recomputes Fidelity, Stability, Counterfactual Actionability, Regulatory Alignment, EES, and rank for every repetition, then reports empirical score uncertainty and rank stability.


In [1]:
# 0. Install and import
!pip -q install shap lime dice-ml xgboost openpyxl
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
from datetime import datetime
import json, random, time, re
import numpy as np, pandas as pd
from scipy.stats import kendalltau
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer
import dice_ml
pd.set_option('display.max_columns',250)
pd.set_option('display.float_format',lambda x:f'{x:.6f}')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.6 MB/s eta 0:00:00


In [2]:
# 1. Locked configuration
DATASET_NAME='compas'; DATASET_LABEL='COMPAS'
N_REPETITIONS=5; TEST_SIZE=0.20
SPLIT_SEEDS=list(range(42,47))
KERNEL_BACKGROUND_SIZE=50; KERNEL_NSAMPLES=500
LIME_NUM_SAMPLES=5000; LIME_NUM_FEATURES=10
N_PERTURBATIONS=30; PERTURBATION_STD=0.05
TOTAL_CFS=3; DICE_METHOD='random'; UNFAVOURABLE_CLASS=1; DESIRED_CLASS=0
IMMUTABLE_FEATURES=['age','sex','race']
REGULATORY_ALIGNMENT_SCORE=1.0
EES_WEIGHTS={'fidelity_score':.25,'stability_score':.25,'actionability_score':.25,'regulatory_alignment_score':.25}
print(SPLIT_SEEDS)


[42, 43, 44, 45, 46]


In [3]:
# 2. Drive and paths
from google.colab import drive
drive.mount('/content/drive')
ROOT_PATH=Path('/content/drive/MyDrive/Master_Research')
PREPROCESSING_DATASETS_PATH=ROOT_PATH/'07_Results'/DATASET_NAME/'02_data_preprocessing_compas'/'datasets'
RESULTS_PATH=ROOT_PATH/'07_Results'/DATASET_NAME/'09_00_ees_uncertainty_rank_stability'
TABLES_PATH=RESULTS_PATH/'tables'; DATASETS_PATH=RESULTS_PATH/'datasets'; LOGS_PATH=RESULTS_PATH/'logs'
for p in [RESULTS_PATH,TABLES_PATH,DATASETS_PATH,LOGS_PATH]: p.mkdir(parents=True,exist_ok=True)
print('Input:',PREPROCESSING_DATASETS_PATH); print('Output:',RESULTS_PATH)


Mounted at /content/drive
Input: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/datasets
Output: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability


In [4]:
# 3. Helpers
def first_existing(folder,names,label):
    for name in names:
        p=folder/name
        if p.exists(): print(label,':',p); return p
    raise FileNotFoundError(label+' not found. Checked:\n'+'\n'.join(str(folder/n) for n in names))
def load_target(path):
    df=pd.read_csv(path)
    if 'target' in df.columns:return df['target'].astype(int)
    if df.shape[1]==1:return df.iloc[:,0].astype(int)
    raise ValueError('Target not identifiable')
def save_df(df,name,folder=TABLES_PATH):
    p=folder/name; df.to_csv(p,index=False); print('Saved:',p); return p
def predict_positive(model,X): return np.asarray(model.predict_proba(X))[:,1]
def inverse_minmax(v):
    v=np.asarray(v,float); lo,hi=np.nanmin(v),np.nanmax(v)
    return np.ones_like(v) if np.isclose(hi,lo) else (hi-v)/(hi-lo)
def safe_tau(a,b):
    if len(a)<2:return 0.0
    t=kendalltau(a,b,nan_policy='omit').statistic
    return 0.0 if pd.isna(t) else float(t)
def clean_lime_name(rule,names):
    m=[f for f in names if f in str(rule)]
    return max(m,key=len) if m else str(rule)
def make_models(seed):
    return {
      'logistic_regression':('Logistic Regression',LogisticRegression(solver='lbfgs',max_iter=2000,random_state=seed)),
      'decision_tree':('Decision Tree',DecisionTreeClassifier(max_depth=5,random_state=seed)),
      'random_forest':('Random Forest',RandomForestClassifier(n_estimators=300,random_state=seed)),
      'xgboost':('XGBoost',XGBClassifier(n_estimators=300,learning_rate=.05,max_depth=3,subsample=.9,colsample_bytree=.9,eval_metric='logloss',random_state=seed,n_jobs=-1)),
      'mlp':('MLP',MLPClassifier(hidden_layer_sizes=(50,),activation='relu',solver='adam',alpha=.0001,max_iter=1000,random_state=seed))}


In [5]:
# 4. Reconstruct full original-space modelling dataset from locked preprocessing outputs
xf1=first_existing(PREPROCESSING_DATASETS_PATH,['X_train_model_original.csv','X_train_original.csv'],'Train features')
xf2=first_existing(PREPROCESSING_DATASETS_PATH,['X_test_model_original.csv','X_test_original.csv'],'Test features')
yf1=first_existing(PREPROCESSING_DATASETS_PATH,['y_train.csv'],'Train target')
yf2=first_existing(PREPROCESSING_DATASETS_PATH,['y_test.csv'],'Test target')
X_full=pd.concat([pd.read_csv(xf1),pd.read_csv(xf2)],ignore_index=True).drop(columns=['target'],errors='ignore')
y_full=pd.concat([load_target(yf1),load_target(yf2)],ignore_index=True)
assert len(X_full)==len(y_full)
print(X_full.shape); display(y_full.value_counts(normalize=True).sort_index())


Train features : /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/datasets/X_train_original.csv
Test features : /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/datasets/X_test_original.csv
Train target : /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/datasets/y_train.csv
Test target : /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/datasets/y_test.csv
(17496, 11)


,proportion
target,
0,0.518919
1,0.481081


In [6]:
# 4B. COMPAS integrity guard
# Repeated splits must use the same retained original-space modelling matrix
# produced by the locked COMPAS preprocessing pipeline.
TARGET_NAME='is_recid'
EXCLUDED_EXACT={'id','name','first','last','age_cat','priors_count.1','c_charge_desc'}
EXCLUDED_PREFIXES=('r_','vr_')
unexpected=[c for c in X_full.columns if c==TARGET_NAME or c in EXCLUDED_EXACT or c.startswith(EXCLUDED_PREFIXES)]
if unexpected:
    raise ValueError(f'COMPAS integrity guard failed; excluded/target columns found: {unexpected}')
required=['age','sex','race']
missing=[c for c in required if c not in X_full.columns]
if missing:
    raise ValueError(f'Required COMPAS immutable/sensitive features missing: {missing}')
if set(pd.Series(y_full).dropna().unique()) - {0,1}:
    raise ValueError(f'COMPAS target must be binary 0/1 after locked cleaning. Found: {sorted(pd.Series(y_full).dropna().unique())}')
print('COMPAS integrity guard passed.')
print('Retained features:',list(X_full.columns))
print('Immutable counterfactual features:',IMMUTABLE_FEATURES)


COMPAS integrity guard passed.
Retained features: ['original_index', 'sex', 'age', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'days_b_screening_arrest', 'c_days_from_compas', 'c_charge_degree']
Immutable counterfactual features: ['age', 'sex', 'race']


In [7]:
# 5. Locked COMPAS preprocessing
# Preserve the retained feature set from the existing COMPAS preprocessing outputs.
# Known retained categorical variables are explicit; any retained object/category/bool
# column is also treated as categorical.
KNOWN_COMPAS_CATEGORICALS=['sex','race','c_charge_degree']
CATEGORICAL_FEATURES=[c for c in X_full.columns if c in KNOWN_COMPAS_CATEGORICALS or str(X_full[c].dtype) in ['object','category','bool']]
NUMERIC_FEATURES=[c for c in X_full.columns if c not in CATEGORICAL_FEATURES]
def make_preprocessor():
    num=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',RobustScaler())])
    cat=Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Missing')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    return ColumnTransformer([('num',num,NUMERIC_FEATURES),('cat',cat,CATEGORICAL_FEATURES)],remainder='drop')
print('Numeric:',NUMERIC_FEATURES); print('Categorical:',CATEGORICAL_FEATURES)


Numeric: ['original_index', 'age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'days_b_screening_arrest', 'c_days_from_compas']
Categorical: ['sex', 'race', 'c_charge_degree']


## 6. Locked EES component logic
**Fidelity** is the equal mean of SHAP reconstruction fidelity, LIME local surrogate fidelity, and SHAP-LIME Kendall agreement. **Stability** uses 30 Gaussian perturbations (SD 0.05) per held-out observation and inverse min-max normalisation across the five models within each repetition. **Actionability** uses DiCE with 3 requested counterfactuals, desired class 0, and age/sex/race immutable. **Regulatory Alignment** remains 1.000 because all four corrected governance-evidence criteria are satisfied. The primary EES uses equal 0.25 weights.


In [8]:
# 6A. Fidelity
def shap_probability(estimator,Xtr,Xte,seed):
    rng=np.random.RandomState(seed); n=min(KERNEL_BACKGROUND_SIZE,len(Xtr)); bg=Xtr[rng.choice(len(Xtr),n,replace=False)]
    if isinstance(estimator,(DecisionTreeClassifier,RandomForestClassifier,XGBClassifier)):
        try:
            ex=shap.TreeExplainer(estimator,data=bg,feature_perturbation='interventional',model_output='probability')
            sv=ex.shap_values(Xte); ev=ex.expected_value
            if isinstance(sv,list): sv=sv[1]
            sv=np.asarray(sv)
            if sv.ndim==3: sv=sv[:,:,1]
            if isinstance(ev,(list,np.ndarray)): ev=np.asarray(ev).reshape(-1)[-1]
            return np.asarray(sv,float),float(ev)
        except Exception: pass
    f=lambda z:estimator.predict_proba(np.asarray(z))[:,1]
    ex=shap.KernelExplainer(f,bg); sv=ex.shap_values(Xte,nsamples=KERNEL_NSAMPLES,silent=True); ev=ex.expected_value
    if isinstance(sv,list): sv=sv[-1]
    sv=np.asarray(sv)
    if sv.ndim==3: sv=sv[:,:,-1]
    if isinstance(ev,(list,np.ndarray)): ev=np.asarray(ev).reshape(-1)[-1]
    return np.asarray(sv,float),float(ev)
def fidelity_for_model(estimator,Xtr,Xte,names,seed):
    sv,ev=shap_probability(estimator,Xtr,Xte,seed); p=predict_positive(estimator,Xte)
    shap_f=float(np.clip(1-np.mean(np.abs(p-(ev+sv.sum(axis=1)))),0,1))
    le=LimeTabularExplainer(Xtr,feature_names=list(names),class_names=['Non-recidivism','Recidivism'],mode='classification',random_state=seed)
    r2=[]; taus=[]
    for i in range(len(Xte)):
        exp=le.explain_instance(Xte[i],estimator.predict_proba,num_features=min(LIME_NUM_FEATURES,len(names)),num_samples=LIME_NUM_SAMPLES)
        r2.append(float(exp.score)); lm={}
        for rule,w in exp.as_list(label=1):
            f=clean_lime_name(rule,names)
            if f in names: lm[f]=abs(float(w))
        common=[f for f in names if f in lm]
        if len(common)>=2:
            sm={names[j]:abs(float(sv[i,j])) for j in range(len(names))}
            taus.append(safe_tau([sm[f] for f in common],[lm[f] for f in common]))
    lime=float(np.clip(np.nanmean(r2),0,1)); tau=float(np.nanmean(taus)) if taus else 0.0; agree=float(np.clip((tau+1)/2,0,1))
    return {'shap_reconstruction_fidelity_score':shap_f,'lime_fidelity_score':lime,'mean_kendall_tau':tau,'shap_lime_agreement_score':agree,'fidelity_score':float(np.mean([shap_f,lime,agree]))}


In [9]:
# 6B. Stability
def raw_stability(estimator,Xte,seed):
    rng=np.random.RandomState(seed); ratios=[]; changes=[]
    for x in Xte:
        x=np.asarray(x,float); p0=estimator.predict_proba(x.reshape(1,-1))[0,1]
        for _ in range(N_PERTURBATIONS):
            noise=rng.normal(0,PERTURBATION_STD,size=x.shape); norm=np.linalg.norm(noise)
            if norm<=0:continue
            pp=estimator.predict_proba((x+noise).reshape(1,-1))[0,1]; ch=abs(pp-p0)
            changes.append(ch); ratios.append(ch/norm)
    return {'mean_output_change':float(np.mean(changes)),'mean_local_lipschitz_ratio':float(np.mean(ratios))}


In [10]:
# 6C. Counterfactual Actionability
def impute_original(Xtr,Xte):
    a=Xtr.copy(); b=Xte.copy()
    for c in NUMERIC_FEATURES:
        med=pd.to_numeric(a[c],errors='coerce').median(); a[c]=pd.to_numeric(a[c],errors='coerce').fillna(med); b[c]=pd.to_numeric(b[c],errors='coerce').fillna(med)
    for c in CATEGORICAL_FEATURES:
        a[c]=a[c].astype(object).where(a[c].notna(),'Missing').astype(str); b[c]=b[c].astype(object).where(b[c].notna(),'Missing').astype(str)
    return a,b
def actionability(full_pipeline,Xtr,Xte,ytr,seed):
    tr,te=impute_original(Xtr,Xte); d=tr.copy(); d['target']=np.asarray(ytr,int)
    data=dice_ml.Data(dataframe=d,continuous_features=NUMERIC_FEATURES,outcome_name='target')
    dm=dice_ml.Model(model=full_pipeline,backend='sklearn',model_type='classifier'); de=dice_ml.Dice(data,dm,method=DICE_METHOD)
    pred=np.asarray(full_pipeline.predict(te),int); pos=np.where(pred==UNFAVOURABLE_CLASS)[0]
    mutable=[c for c in tr.columns if c not in IMMUTABLE_FEATURES]
    ranges={c:max(float(tr[c].max()-tr[c].min()),1e-12) for c in NUMERIC_FEATURES}
    found=[]; mind=[]; minchg=[]
    for k in pos:
        original=te.iloc[k]; query=te.iloc[[k]].copy()
        try:
            res=de.generate_counterfactuals(query,total_CFs=TOTAL_CFS,desired_class=DESIRED_CLASS,features_to_vary=mutable,random_seed=seed,verbose=False)
            cf=res.cf_examples_list[0].final_cfs_df
            if cf is None or cf.empty: found.append(0); continue
            found.append(1); cf=cf.drop(columns=['target'],errors='ignore'); ds=[]; cs=[]
            for _,r in cf.iterrows():
                ds.append(np.mean([abs(float(r[c])-float(original[c]))/ranges[c] for c in NUMERIC_FEATURES]))
                n=0
                for c in tr.columns:
                    try:same=np.isclose(float(original[c]),float(r[c]))
                    except:same=str(original[c])==str(r[c])
                    if not same:n+=1
                cs.append(n)
            mind.append(min(ds)); minchg.append(min(cs))
        except Exception: found.append(0)
    if len(pos)==0:return {'availability_score':np.nan,'proximity_score':np.nan,'sparsity_score':np.nan,'actionability_score':np.nan,'n_unfavourable_predictions':0}
    av=float(np.mean(found)); pr=float(np.clip(1-np.mean(mind),0,1)) if mind else 0.0; sp=float(np.clip(1-np.mean(minchg)/tr.shape[1],0,1)) if minchg else 0.0
    return {'availability_score':av,'proximity_score':pr,'sparsity_score':sp,'actionability_score':float(np.mean([av,pr,sp])),'n_unfavourable_predictions':int(len(pos))}


In [11]:
# 7. Optimised/restart-safe repeated stratified 80/20 EES evaluation
# ------------------------------------------------------------------
# IMPORTANT:
# - Same methodology as original analysis.
# - Checkpoints AFTER EVERY MODEL rather than after every repetition.
# - Automatically skips model/seed combinations already completed.
# - Safe to rerun after timeout/interruption.

import time
import random
import numpy as np
import pandas as pd
from pathlib import Path

CHECKPOINT_PATH = DATASETS_PATH / "ees_uncertainty_model_checkpoint.csv"

# ---------------------------------------------------------------
# 1. Load any previously completed work
# ---------------------------------------------------------------
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)

    completed_keys = set(
        zip(
            checkpoint_df["split_seed"].astype(int),
            checkpoint_df["model_id"].astype(str)
        )
    )

    all_rows = checkpoint_df.to_dict("records")

    print(
        f"Checkpoint loaded: {len(checkpoint_df)} model evaluations "
        f"already completed."
    )
else:
    checkpoint_df = pd.DataFrame()
    completed_keys = set()
    all_rows = []

    print("No model-level checkpoint found. Starting from scratch.")

run_log = []

# ---------------------------------------------------------------
# 2. Repeated stratified holdout
# ---------------------------------------------------------------
for repetition, seed in enumerate(SPLIT_SEEDS, 1):

    print("\n" + "=" * 100)
    print(
        f"REPETITION {repetition}/{N_REPETITIONS} | "
        f"seed={seed}"
    )
    print("=" * 100)

    repetition_start = time.time()

    np.random.seed(seed)
    random.seed(seed)

    # -----------------------------------------------------------
    # Split
    # -----------------------------------------------------------
    Xtr, Xte, ytr, yte = train_test_split(
        X_full,
        y_full,
        test_size=TEST_SIZE,
        stratify=y_full,
        random_state=seed
    )

    Xtr = Xtr.reset_index(drop=True)
    Xte = Xte.reset_index(drop=True)
    ytr = ytr.reset_index(drop=True)
    yte = yte.reset_index(drop=True)

    # -----------------------------------------------------------
    # Preprocessing is performed ONCE per split
    # -----------------------------------------------------------
    prep = make_preprocessor()

    A = np.asarray(
        prep.fit_transform(Xtr),
        dtype=float
    )

    B = np.asarray(
        prep.transform(Xte),
        dtype=float
    )

    names = [
        f.replace("num__", "").replace("cat__", "")
        for f in prep.get_feature_names_out()
    ]

    models = make_models(seed)

    # -----------------------------------------------------------
    # 3. Model-level evaluation
    # -----------------------------------------------------------
    for mid, (mname, est) in models.items():

        key = (int(seed), str(mid))

        # Skip completed model/seed combinations
        if key in completed_keys:
            print(f"  ✓ {mname}: already completed — skipped")
            continue

        print(f"\n  → {mname}")
        model_start = time.time()

        # ---------------- FIT ----------------
        t0 = time.time()

        est.fit(A, ytr)

        fit_seconds = time.time() - t0
        print(f"      Fit            : {fit_seconds:8.1f}s")

        # Full pipeline required by actionability()
        full = Pipeline([
            ("preprocessor", prep),
            ("classifier", est)
        ])

        # ---------------- FIDELITY ----------------
        t0 = time.time()

        f = fidelity_for_model(
            est,
            A,
            B,
            names,
            seed
        )

        fidelity_seconds = time.time() - t0
        print(f"      Fidelity       : {fidelity_seconds:8.1f}s")

        # ---------------- STABILITY ----------------
        t0 = time.time()

        s = raw_stability(
            est,
            B,
            seed
        )

        stability_seconds = time.time() - t0
        print(f"      Stability      : {stability_seconds:8.1f}s")

        # ---------------- ACTIONABILITY ----------------
        t0 = time.time()

        a = actionability(
            full,
            Xtr,
            Xte,
            ytr,
            seed
        )

        actionability_seconds = time.time() - t0
        print(f"      Actionability  : {actionability_seconds:8.1f}s")

        total_seconds = time.time() - model_start

        # -------------------------------------------------------
        # Store raw component results
        # Stability normalisation/EES ranking happen later because
        # all five models are required for candidate-relative scaling.
        # -------------------------------------------------------
        row = {
            "dataset_name": DATASET_NAME,
            "dataset_label": DATASET_LABEL,
            "repetition": repetition,
            "split_seed": seed,
            "model_id": mid,
            "model_name": mname,

            **f,
            **s,
            **a,

            "regulatory_alignment_score":
                REGULATORY_ALIGNMENT_SCORE,

            "fit_seconds": fit_seconds,
            "fidelity_seconds": fidelity_seconds,
            "stability_seconds": stability_seconds,
            "actionability_seconds": actionability_seconds,
            "total_seconds": total_seconds
        }

        all_rows.append(row)

        completed_keys.add(key)

        # -------------------------------------------------------
        # CHECKPOINT AFTER EVERY MODEL
        # -------------------------------------------------------
        pd.DataFrame(all_rows).to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        print(
            f"      Total          : {total_seconds:8.1f}s"
        )
        print("      ✓ checkpoint saved")

    print(
        f"\nRepetition processing time: "
        f"{(time.time() - repetition_start)/60:.1f} min"
    )


# =================================================================
# 4. Reconstruct completed repetitions
# =================================================================

raw_results_df = pd.DataFrame(all_rows)

# Remove accidental duplicate model/seed rows if cell was restarted
raw_results_df = (
    raw_results_df
    .drop_duplicates(
        subset=["split_seed", "model_id"],
        keep="last"
    )
    .sort_values(
        ["repetition", "model_id"]
    )
    .reset_index(drop=True)
)

# Identify repetitions containing ALL FIVE models
counts = (
    raw_results_df
    .groupby("split_seed")["model_id"]
    .nunique()
)

complete_seeds = counts[counts == 5].index.tolist()

print("\n" + "=" * 100)
print(
    f"Complete repetitions: "
    f"{len(complete_seeds)}/{N_REPETITIONS}"
)
print("=" * 100)

# Only complete repetitions can be used for relative stability
# normalisation and EES ranking.
uncertainty_results_df = raw_results_df[
    raw_results_df["split_seed"].isin(complete_seeds)
].copy()


# =================================================================
# 5. Calculate candidate-relative Stability + EES PER SPLIT
# =================================================================

processed_repetitions = []

for seed, rdf in uncertainty_results_df.groupby(
    "split_seed",
    sort=True
):

    rdf = rdf.copy()

    rdf["stability_score"] = inverse_minmax(
        rdf["mean_local_lipschitz_ratio"]
    )

    rdf["ees_score"] = (
        0.25 * rdf["fidelity_score"]
        + 0.25 * rdf["stability_score"]
        + 0.25 * rdf["actionability_score"]
        + 0.25 * rdf["regulatory_alignment_score"]
    )

    rdf["ees_rank"] = (
        rdf["ees_score"]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )

    processed_repetitions.append(rdf)


if processed_repetitions:

    uncertainty_results_df = pd.concat(
        processed_repetitions,
        ignore_index=True
    )

else:

    uncertainty_results_df = pd.DataFrame()


# =================================================================
# 6. Save processed results
# =================================================================

FINAL_RESULTS_PATH = (
    DATASETS_PATH /
    "ees_uncertainty_repetition_results.csv"
)

uncertainty_results_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

print(
    f"\nRows available for uncertainty analysis: "
    f"{len(uncertainty_results_df)}"
)

print(
    f"Expected when complete: "
    f"{N_REPETITIONS * 5}"
)

print(
    f"\nProcessed results saved to:\n"
    f"{FINAL_RESULTS_PATH}"
)

Checkpoint loaded: 25 model evaluations already completed.

REPETITION 1/5 | seed=42
  ✓ Logistic Regression: already completed — skipped
  ✓ Decision Tree: already completed — skipped
  ✓ Random Forest: already completed — skipped
  ✓ XGBoost: already completed — skipped
  ✓ MLP: already completed — skipped

Repetition processing time: 0.0 min

REPETITION 2/5 | seed=43
  ✓ Logistic Regression: already completed — skipped
  ✓ Decision Tree: already completed — skipped
  ✓ Random Forest: already completed — skipped
  ✓ XGBoost: already completed — skipped
  ✓ MLP: already completed — skipped

Repetition processing time: 0.0 min

REPETITION 3/5 | seed=44
  ✓ Logistic Regression: already completed — skipped
  ✓ Decision Tree: already completed — skipped
  ✓ Random Forest: already completed — skipped
  ✓ XGBoost: already completed — skipped
  ✓ MLP: already completed — skipped

Repetition processing time: 0.0 min

REPETITION 4/5 | seed=45
  ✓ Logistic Regression: already completed — skippe

In [12]:
# 8. Reviewer-facing uncertainty summary
rows=[]
for mid,g in uncertainty_results_df.groupby('model_id'):
    rows.append({'dataset_name':DATASET_NAME,'dataset_label':DATASET_LABEL,'model_id':mid,'model_name':g.model_name.iloc[0],'n_repetitions':len(g),
      'mean_fidelity':g.fidelity_score.mean(),'sd_fidelity':g.fidelity_score.std(ddof=1),
      'mean_stability':g.stability_score.mean(),'sd_stability':g.stability_score.std(ddof=1),
      'mean_actionability':g.actionability_score.mean(),'sd_actionability':g.actionability_score.std(ddof=1),
      'mean_regulatory_alignment':g.regulatory_alignment_score.mean(),'sd_regulatory_alignment':g.regulatory_alignment_score.std(ddof=1),
      'mean_ees':g.ees_score.mean(),'sd_ees':g.ees_score.std(ddof=1),'ees_empirical_2_5_pct':g.ees_score.quantile(.025),'ees_empirical_97_5_pct':g.ees_score.quantile(.975),'ees_min':g.ees_score.min(),'ees_max':g.ees_score.max(),
      'mean_rank':g.ees_rank.mean(),'median_rank':g.ees_rank.median(),'best_rank':g.ees_rank.min(),'worst_rank':g.ees_rank.max(),'rank_1_count':int((g.ees_rank==1).sum()),'rank_1_frequency':float((g.ees_rank==1).mean())})
uncertainty_summary_df=pd.DataFrame(rows).sort_values(['mean_rank','mean_ees'],ascending=[True,False]).reset_index(drop=True)
display(uncertainty_summary_df)


,dataset_name,dataset_label,model_id,model_name,n_repetitions,mean_fidelity,sd_fidelity,mean_stability,sd_stability,mean_actionability,sd_actionability,mean_regulatory_alignment,sd_regulatory_alignment,mean_ees,sd_ees,ees_empirical_2_5_pct,ees_empirical_97_5_pct,ees_min,ees_max,mean_rank,median_rank,best_rank,worst_rank,rank_1_count,rank_1_frequency
0,compas,COMPAS,decision_tree,Decision Tree,5,0.643052,0.014882,1.000000,0.000000,0.946394,0.004042,1.000000,0.000000,0.897361,0.004543,0.893438,0.903943,0.893403,0.904530,1.000000,1.000000,1,1,5,1.000000
1,compas,COMPAS,mlp,MLP,5,0.684917,0.045020,0.583315,0.184100,0.947454,0.003098,1.000000,0.000000,0.803922,0.044428,0.734282,0.830246,0.726069,0.830357,2.600000,2.000000,2,4,0,0.000000
2,compas,COMPAS,xgboost,XGBoost,5,0.699089,0.008281,0.635494,0.020946,0.947864,0.003362,1.000000,0.000000,0.820612,0.006060,0.812081,0.825623,0.811489,0.825685,2.800000,3.000000,2,4,0,0.000000
3,compas,COMPAS,logistic_regression,Logistic Regression,5,0.588632,0.022025,0.705320,0.022775,0.948301,0.004039,1.000000,0.000000,0.810563,0.007938,0.799579,0.818735,0.798713,0.819060,3.600000,4.000000,3,4,0,0.000000
4,compas,COMPAS,random_forest,Random Forest,5,0.702573,0.012607,0.000000,0.000000,0.947175,0.002907,1.000000,0.000000,0.662437,0.002895,0.659333,0.665635,0.659258,0.665702,5.000000,5.000000,5,5,0,0.000000


In [13]:
# 9. Rank distributions and champions
rank_distribution_df=uncertainty_results_df.groupby(['model_id','model_name','ees_rank']).size().rename('count').reset_index(); rank_distribution_df['frequency']=rank_distribution_df['count']/N_REPETITIONS
champions_df=uncertainty_results_df[uncertainty_results_df.ees_rank==1][['repetition','split_seed','model_id','model_name','ees_score']].sort_values(['repetition','model_id']).reset_index(drop=True)
champion_frequency_df=champions_df.groupby(['model_id','model_name']).size().rename('rank_1_count').reset_index(); champion_frequency_df['rank_1_frequency']=champion_frequency_df['rank_1_count']/N_REPETITIONS
champion_frequency_df=champion_frequency_df.sort_values('rank_1_frequency',ascending=False).reset_index(drop=True)
display(champion_frequency_df)


,model_id,model_name,rank_1_count,rank_1_frequency
0,decision_tree,Decision Tree,5,1.000000


In [15]:
# 10. Save outputs

# Core outputs
outputs_to_save = [
    (uncertainty_results_df,
     'ees_uncertainty_repetition_results.csv',
     DATASETS_PATH),

    (uncertainty_results_df,
     'ees_uncertainty_repetition_results.csv',
     TABLES_PATH),

    (uncertainty_summary_df,
     'ees_uncertainty_summary.csv',
     TABLES_PATH),

    (rank_distribution_df,
     'ees_rank_distribution_long.csv',
     TABLES_PATH),

    (champions_df,
     'ees_split_champions.csv',
     TABLES_PATH),

    (champion_frequency_df,
     'ees_rank1_frequency.csv',
     TABLES_PATH)
]

for df, name, folder in outputs_to_save:
    save_df(df, name, folder)


# Save run log only if it exists
if 'run_log_df' in globals():
    save_df(
        run_log_df,
        'ees_uncertainty_run_log.csv',
        LOGS_PATH
    )
else:
    print("Note: run_log_df was not created; skipping run-log CSV.")


# Excel workbook
excel = TABLES_PATH / 'ees_uncertainty_rank_stability_outputs.xlsx'

with pd.ExcelWriter(excel, engine='openpyxl') as w:

    uncertainty_results_df.to_excel(
        w,
        sheet_name='repetition_results',
        index=False
    )

    uncertainty_summary_df.to_excel(
        w,
        sheet_name='uncertainty_summary',
        index=False
    )

    rank_distribution_df.to_excel(
        w,
        sheet_name='rank_distribution',
        index=False
    )

    champion_frequency_df.to_excel(
        w,
        sheet_name='rank1_frequency',
        index=False
    )

    champions_df.to_excel(
        w,
        sheet_name='split_champions',
        index=False
    )

    if 'run_log_df' in globals():
        run_log_df.to_excel(
            w,
            sheet_name='run_log',
            index=False
        )


# Reproducibility manifest
manifest = {
    'dataset_name': DATASET_NAME,
    'dataset_label': DATASET_LABEL,
    'notebook': '09_00_ees_uncertainty_rank_stability_compas.ipynb',
    'run_timestamp': datetime.now().isoformat(),
    'n_repetitions': N_REPETITIONS,
    'test_size': TEST_SIZE,
    'stratified': True,
    'split_seeds': SPLIT_SEEDS,
    'models': list(make_models(42).keys()),
    'primary_ees_weights': EES_WEIGHTS,
    'regulatory_alignment_score': REGULATORY_ALIGNMENT_SCORE,
    'interpretation_note': (
        'The 2.5th-97.5th percentile range is an empirical uncertainty '
        'interval across overlapping repeated stratified holdouts, not an '
        'independent-sample population confidence interval.'
    )
}

with open(
    LOGS_PATH / 'ees_uncertainty_rank_stability_manifest.json',
    'w'
) as f:
    json.dump(manifest, f, indent=4)

print('Saved:', excel)

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/datasets/ees_uncertainty_repetition_results.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/tables/ees_uncertainty_repetition_results.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/tables/ees_uncertainty_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/tables/ees_rank_distribution_long.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/tables/ees_split_champions.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/tables/ees_rank1_frequency.csv
Note: run_log_df was not created; skipping run-log CSV.
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/09_00_ees_uncertainty_rank_stability/tables/e

## Reviewer-facing interpretation
Use this analysis together with the existing robustness work: **10-fold stratified CV** addresses predictive robustness; the existing **EES weighting sensitivity** addresses sensitivity to evaluation priorities; this notebook addresses **sampling/split uncertainty in EES components and model ranking**.

Report mean EES, SD, the empirical 2.5th–97.5th percentile interval, mean/median rank, best/worst rank, and rank-1 frequency. Because repeated holdouts overlap in observations, do not describe the 20 repetitions as 20 independent datasets.
